In [1]:
# Here, we are saving the output value of the root finding function
# for different values of (N_tilde and t_Q) in a csv file

import numpy as np
import pandas as pd
from scipy.optimize import root_scalar as rs
import astropy
from astropy.cosmology import FlatLambdaCDM

Planck = astropy.cosmology.realizations.Planck18

# T_CMB = Planck.Tcmb0  # Temperature of the CMB
# H_0 = Planck.H0  # Current Hubble constant
# Omega_m = Planck.Om0  # Matter density parameter
# Omega_lambda = Planck.Ode0  # Dark energy density parameter
c = astropy.constants.c.to('km/s')  # Speed of light in km/s
# sigma_T = astropy.constants.sigma_T.to('km2')  # Thomson scattering cross-section in km^2
Omega_b = Planck.Ob0  # Baryon density parameter
# h = Planck.h  # Dimensionless Hubble parameter
rho_crit_0 = Planck.critical_density0.to('kg/m3')  # Critical density of the universe at z = 0 in kg/m^3
z = 8

In [11]:
# Setting up values and limits

N_tilde = np.linspace(0.1, 100, int(100))  # N_tilde: 0 to 100
t_Q = np.linspace(0.1, 100, int(100))  # t_Q: 0 to 100

# Setting up constants
n_H = rho_crit_0/(astropy.constants.m_p) * (1 - 0.247) * (Omega_b) * (1 + z)**3
n_H = n_H.to('km-3')

In [12]:
# Defining the root finding function

def radius_calc(theta, n_H): 
    N_tilde, t_tilde = theta
    return (((3)/(4*np.pi*n_H))**(1/3)) * (N_tilde * t_tilde)**(1/3) * (1e58)**(1/3) * (1e7 * 3.156e7)**(1/3)  

def root_func(t, theta, n_H, c = c):
    N_tilde, t_tilde = theta
    N_dot = N_tilde * 1e58  # in s^-1
    t_Q = t_tilde * 1e7 * 3.156e7  # in seconds

    return (t * 1e7 * 3.156e7) + (radius_calc([N_tilde, t], n_H).value)/(c.value) - t_Q

In [13]:
# Finding the root for each combination of N_tilde and t_Q and saving the results in a csv file
# adding a progress bar using tqdm
from tqdm import tqdm
results = []
for N in tqdm(N_tilde):
    for t in tqdm(t_Q):
        root = rs(root_func, args=([N, t], n_H), bracket=[1e-40, 1e40]).root
        results.append((N, t, root))

results

  0%|          | 0/100 [00:00<?, ?it/s]

100%|██████████| 100/100 [00:11<00:00,  8.38it/s]


[(np.float64(0.1), np.float64(0.1), 0.0014853230188977059),
 (np.float64(0.1), np.float64(1.1090909090909091), 0.44828011810269175),
 (np.float64(0.1), np.float64(2.118181818181818), 1.2005181393927238),
 (np.float64(0.1), np.float64(3.1272727272727274), 2.0333969123541618),
 (np.float64(0.1), np.float64(4.136363636363636), 2.904445842854833),
 (np.float64(0.1), np.float64(5.145454545454545), 3.798282521481327),
 (np.float64(0.1), np.float64(6.154545454545454), 4.7074740989567045),
 (np.float64(0.1), np.float64(7.163636363636363), 5.627815431341045),
 (np.float64(0.1), np.float64(8.172727272727272), 6.556675750064841),
 (np.float64(0.1), np.float64(9.181818181818182), 7.492289807677322),
 (np.float64(0.1), np.float64(10.19090909090909), 8.433410266333704),
 (np.float64(0.1), np.float64(11.2), 9.379119978327516),
 (np.float64(0.1), np.float64(12.209090909090909), 10.328722946176052),
 (np.float64(0.1), np.float64(13.218181818181817), 11.281677256902539),
 (np.float64(0.1), np.float64(14

In [14]:
# Converting the results to a pandas dataframe
df = pd.DataFrame(results, columns=['N_tilde', 't_Q', 'root'])
# Saving the dataframe to a csv file
df.to_csv('root_finding_results_shrink.csv', index=False)
 

print(df)

      N_tilde         t_Q       root
0         0.1    0.100000   0.001485
1         0.1    1.109091   0.448280
2         0.1    2.118182   1.200518
3         0.1    3.127273   2.033397
4         0.1    4.136364   2.904446
...       ...         ...        ...
9995    100.0   95.963636  61.822621
9996    100.0   96.972727  62.675440
9997    100.0   97.981818  63.529455
9998    100.0   98.990909  64.384644
9999    100.0  100.000000  65.240984

[10000 rows x 3 columns]


In [16]:
rs(root_func, args=([100, 96.5], n_H), bracket=[1e-40, 1e40]).root

62.27577111360941